# CGRU-5mC V9 — v5e-1 TPU / JAX / Fold ~97% + Full-Development Ensemble / Fixed 0.50

V9 gives the CV folds slightly more room to reach ~96–97% training accuracy, while OOF predictions still use the best-validation checkpoints. After CV, it trains a small ensemble on the full development set and blends it with the five best-validation fold models for the independent prediction. Threshold remains fixed at 0.50.

Important distinction:

- `best_train.weights.h5` is selected by **training accuracy** and is saved so the training run can reach/retain its highest training accuracy.
- `best_val.weights.h5` is selected independently by **validation accuracy** and is the model used for OOF and independent-test predictions.
- The decision threshold is **fixed at 0.50 everywhere**.
- The independent test set is never used for threshold selection or checkpoint selection.

Expected Drive folder:

```text
My Drive/5Mc/
├── train_positive_data.csv
├── train_negative_data.csv
├── test_positive_data.csv
├── test_negative_data.csv
├── train_full_clean.csv
└── test_full_clean.csv
```

`PRIMARY` uses the four split class files.  
`SECONDARY` uses the two `full_clean` files.

In [ ]:
# ============================================================
# CELL 0 — SAFE INSTALLATION FOR COLAB v5e-1 TPU
# ============================================================
#
# IMPORTANT:
# - Select Runtime -> Change runtime type -> TPU BEFORE running.
# - Do NOT install/upgrade TensorFlow.
# - Do NOT install/upgrade JAX/JAXLIB in a Colab TPU runtime.
#   Colab's TPU-specific JAX stack must remain intact.
# ============================================================

import os
import sys
import subprocess
import importlib.util

os.environ["KERAS_BACKEND"] = "jax"

def ensure_package(import_name, pip_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        )
    else:
        print(f"{import_name}: already available")

# These are safe utility packages.
ensure_package("numpy", "numpy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("matplotlib", "matplotlib")
ensure_package("h5py", "h5py")

# Keras is only installed if it is absent. We intentionally do NOT upgrade it.
if importlib.util.find_spec("keras") is None:
    print("Keras is missing; installing Keras 3 without touching JAX.")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "keras>=3.10,<4"]
    )
else:
    print("keras: already available")

if importlib.util.find_spec("jax") is None:
    raise RuntimeError(
        "JAX is missing. Do not pip-install JAX manually on the TPU runtime. "
        "Reconnect to a fresh Colab v5e-1 TPU runtime."
    )

print("\nSafe installation check complete.")
print("KERAS_BACKEND =", os.environ["KERAS_BACKEND"])

numpy: already available
pandas: already available
sklearn: already available
matplotlib: already available
h5py: already available
keras: already available

Safe installation check complete.
KERAS_BACKEND = jax


In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

DATASET_MODE = "SECONDARY"       # "PRIMARY" or "SECONDARY"

DRIVE_DATA_DIR = "/content/drive/MyDrive/5Mc"
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05"

SEED = 42
N_FOLDS = 5

SEQUENCE_LENGTH = 41
CENTER_INDEX = 20
AMBIGUOUS_IUPAC = "NRYWSKMBDHV"

# ------------------------------------------------------------
# FIXED DECISION THRESHOLD
# ------------------------------------------------------------
FIXED_THRESHOLD = 0.50

# ------------------------------------------------------------
# V8: TARGET ~96–97% TRAIN ACCURACY WITH VALIDATION CONTROL
# ------------------------------------------------------------
# Natural class distribution is intentionally preserved.
# This avoids probability shift caused by aggressive positive oversampling.
TRAIN_BATCH_SIZE = 512
EVAL_BATCH_SIZE = 2048

MAX_EPOCHS = 55

INITIAL_LR = 3.5e-4
MIN_LR = 5e-7
WEIGHT_DECAY = 3e-6
LABEL_SMOOTHING = 0.0

# Slightly lighter regularization than V7 so training can move toward 96–97%.
SPATIAL_DROPOUT = 0.05
ATTENTION_DROPOUT = 0.05
DROPOUT = 0.12

# Mid-high model capacity: stronger than V7, still below the V6 overfitting setup.
BRANCH_FILTERS = 88
DILATION_RATES = (1, 2, 4)
CBAM_REDUCTION = 8

BIGRU_UNITS = 160
MHA_HEADS = 4
MHA_KEY_DIM = 64
FFN_UNITS = 352

DENSE_1 = 224
DENSE_2 = 112

USE_DUAL_GLOBAL_POOLING = True

# ------------------------------------------------------------
# V9 FINAL FULL-DEVELOPMENT ENSEMBLE
# ------------------------------------------------------------
# The final full-development models are trained without touching the test set.
FINAL_ENSEMBLE_SEEDS = (42, 52, 62)

# Stop each full-development model once it has fitted the whole development
# set to this band, or when the CV-derived epoch cap is reached.
FINAL_TARGET_TRAIN_ACCURACY = 0.965
FINAL_MIN_EPOCHS = 25
FINAL_MAX_EPOCHS = 30

# Predeclared blend; do not change this after looking at the test result.
# Fold ensemble = strong generalization anchor.
# Full-development ensemble = uses 100% of development data.
FINAL_BLEND_CV_WEIGHT = 0.60
FINAL_BLEND_FULLDEV_WEIGHT = 0.40

# Training stops only if training loss genuinely stalls.
USE_TRAIN_LOSS_EARLY_STOPPING = False
TRAIN_LOSS_PATIENCE = 8
LR_PATIENCE = 2
FOLD_EARLY_STOP_PATIENCE = 14

# Exact duplicate sequences are assigned to the same CV fold.
RUN_EXACT_TRAIN_TEST_OVERLAP_AUDIT = True

CACHE_PREPARED_TO_DRIVE = True
FORCE_REBUILD_PREPARED = False

print("Configuration loaded.")
print("Fixed threshold:", FIXED_THRESHOLD)

Configuration loaded.
Fixed threshold: 0.5


In [ ]:
# ============================================================
# CELL 2 — JAX / KERAS / v5e-1 TPU INITIALIZATION
# ============================================================

import os

# ------------------------------------------------------------
# Fallback configuration
# ------------------------------------------------------------
# Normally SEED comes from Cell 1.
# If Cell 1 was not run for some reason, use 42.
SEED = globals().get("SEED", 42)

# Must be set BEFORE importing keras
os.environ["KERAS_BACKEND"] = "jax"

# ------------------------------------------------------------
# JAX / TPU CHECK
# ------------------------------------------------------------
import jax

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())
print("JAX device count:", jax.device_count())

if not any("TpuDevice" in repr(d) for d in jax.devices()):
    raise RuntimeError(
        "JAX cannot see a TPU.\n"
        f"Current devices: {jax.devices()}\n\n"
        "Go to:\n"
        "Runtime -> Disconnect and delete runtime\n"
        "Then select v5e-1 TPU and run again from Cell 0."
    )

# ------------------------------------------------------------
# KERAS WITH JAX BACKEND
# ------------------------------------------------------------
import keras
from keras import layers
from keras import ops

if keras.backend.backend() != "jax":
    raise RuntimeError(
        f"Keras backend is '{keras.backend.backend()}', expected 'jax'.\n"
        "Restart the runtime and run again from Cell 0."
    )

# TPU-friendly mixed precision
keras.mixed_precision.set_global_policy("mixed_bfloat16")

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------
from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# GENERAL IMPORTS
# ------------------------------------------------------------
import gc
import re
import json
import math
import time
import shutil
import random
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
)

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# STATUS
# ------------------------------------------------------------
print("\n========================================")
print("TPU / JAX INITIALIZATION SUCCESSFUL")
print("========================================")

print("SEED:", SEED)
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())
print("JAX:", jax.__version__)
print("Devices:", jax.devices())
print("Device count:", jax.device_count())
print(
    "Precision policy:",
    keras.mixed_precision.global_policy()
)

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version: 0.7.2
JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
JAX device count: 1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

TPU / JAX INITIALIZATION SUCCESSFUL
SEED: 42
Keras: 3.13.2
Backend: jax
JAX: 0.7.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device count: 1
Precision policy: <DTypePolicy "mixed_bfloat16">


In [ ]:
# ============================================================
# CELL 3 — DATASET FILES + HELPERS
# ============================================================

DATASET_MODE = DATASET_MODE.upper().strip()

if DATASET_MODE not in {"PRIMARY", "SECONDARY"}:
    raise ValueError("DATASET_MODE must be PRIMARY or SECONDARY.")

DATA_DIR = Path(DRIVE_DATA_DIR)

FILES = {
    "train_pos": DATA_DIR / "train_positive_data.csv",
    "train_neg": DATA_DIR / "train_negative_data.csv",
    "test_pos": DATA_DIR / "test_positive_data.csv",
    "test_neg": DATA_DIR / "test_negative_data.csv",
    "train_full": DATA_DIR / "train_full_clean.csv",
    "test_full": DATA_DIR / "test_full_clean.csv",
}

if DATASET_MODE == "PRIMARY":
    TRAIN_SOURCES = [
        (FILES["train_pos"], 1),
        (FILES["train_neg"], 0),
    ]
    TEST_SOURCES = [
        (FILES["test_pos"], 1),
        (FILES["test_neg"], 0),
    ]
else:
    TRAIN_SOURCES = [
        (FILES["train_full"], None),
    ]
    TEST_SOURCES = [
        (FILES["test_full"], None),
    ]

for path, _ in TRAIN_SOURCES + TEST_SOURCES:
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset file: {path}")

RESULT_DIR = Path(DRIVE_RESULTS_ROOT) / DATASET_MODE
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_PREP_DIR = Path(f"/content/cgru5mc_v9_{DATASET_MODE.lower()}_prepared")
DRIVE_PREP_DIR = DATA_DIR / f"_prepared_cgru5mc_v9_{DATASET_MODE.lower()}"

SEQ_CANDIDATES = {
    "sequence", "seq", "dna", "dna_sequence", "dna sequence",
    "fragment", "promoter_sequence", "promoter sequence"
}

LABEL_CANDIDATES = {
    "label", "target", "class", "y", "status",
    "methylation", "methylated", "is_5mc", "5mc"
}

ALLOWED_RE = re.compile(
    rf"^[ACGT{re.escape(AMBIGUOUS_IUPAC)}]+$"
)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def conv(x):
        if isinstance(x, np.integer):
            return int(x)
        if isinstance(x, np.floating):
            return float(x)
        if isinstance(x, np.ndarray):
            return x.tolist()
        return str(x)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=conv)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_sequences(series):
    return (
        series.astype(str)
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
        .str.strip()
    )

def inspect_columns(path):
    sample = pd.read_csv(path, nrows=100)

    lower = {
        str(c).strip().lower(): c
        for c in sample.columns
    }

    seq_col = next(
        (lower[k] for k in SEQ_CANDIDATES if k in lower),
        None
    )

    label_col = next(
        (lower[k] for k in LABEL_CANDIDATES if k in lower),
        None
    )

    if seq_col is None:
        for c in sample.columns:
            s = normalize_sequences(sample[c])
            valid_ratio = s.map(
                lambda x: bool(ALLOWED_RE.fullmatch(x))
            ).mean()
            median_len = s.str.len().median()

            if valid_ratio >= 0.80 and median_len == SEQUENCE_LENGTH:
                seq_col = c
                break

    if seq_col is None:
        raise ValueError(
            f"Could not identify the sequence column in {path}. "
            f"Columns={list(sample.columns)}"
        )

    return seq_col, label_col

def parse_binary_labels(series):
    if pd.api.types.is_numeric_dtype(series):
        y = pd.to_numeric(series, errors="raise").to_numpy()

        if not set(np.unique(y)).issubset({0, 1, 0.0, 1.0}):
            raise ValueError("Combined file contains non-binary labels.")

        return y.astype(np.uint8)

    s = series.astype(str).str.strip().str.lower()

    positive = {
        "1", "positive", "pos", "5mc", "methylated", "true"
    }

    negative = {
        "0", "negative", "neg", "non-5mc",
        "non5mc", "unmethylated", "false"
    }

    bad = ~(s.isin(positive) | s.isin(negative))

    if bad.any():
        raise ValueError(f"Unknown labels: {s[bad].unique()[:20]}")

    y = np.zeros(len(s), dtype=np.uint8)
    y[s.isin(positive).to_numpy()] = 1
    return y

def source_signature():
    def signature_one(path, fixed):
        st = Path(path).stat()
        return {
            "path": str(path),
            "size": int(st.st_size),
            "mtime": int(st.st_mtime),
            "fixed_label": fixed,
        }

    return {
        "dataset_mode": DATASET_MODE,
        "train": [
            signature_one(path, fixed)
            for path, fixed in TRAIN_SOURCES
        ],
        "test": [
            signature_one(path, fixed)
            for path, fixed in TEST_SOURCES
        ],
        "sequence_length": SEQUENCE_LENGTH,
        "n_folds": N_FOLDS,
        "ambiguous_iupac": AMBIGUOUS_IUPAC,
    }

print("Result directory:", RESULT_DIR)

Result directory: /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY


In [ ]:
# ============================================================
# CELL 4 — PREPARE COMPACT uint8 DATA + HASH-BASED CV FOLDS
# ============================================================

# A=0, C=1, G=2, T=3, IUPAC ambiguity=4
ASCII_TO_CODE = np.full(256, 4, dtype=np.uint8)
ASCII_TO_CODE[ord("A")] = 0
ASCII_TO_CODE[ord("C")] = 1
ASCII_TO_CODE[ord("G")] = 2
ASCII_TO_CODE[ord("T")] = 3

def cache_valid(folder):
    manifest = load_json(Path(folder) / "manifest.json", None)
    return (
        manifest is not None
        and manifest.get("source_signature") == source_signature()
    )

def encode_sequence_chunk(seqs):
    raw = "".join(seqs.tolist()).encode("ascii")
    ascii_arr = np.frombuffer(
        raw, dtype=np.uint8
    ).reshape(len(seqs), SEQUENCE_LENGTH)

    return ASCII_TO_CODE[ascii_arr]

def append_binary(handle, arr):
    np.asarray(arr).tofile(handle)

def build_prepared(folder):
    folder = Path(folder)

    if folder.exists():
        shutil.rmtree(folder)

    folder.mkdir(parents=True, exist_ok=True)

    binary_paths = {
        "train_codes": folder / "train_codes.bin",
        "train_labels": folder / "train_labels.bin",
        "train_folds": folder / "train_folds.bin",
        "train_hashes": folder / "train_hashes.bin",
        "test_codes": folder / "test_codes.bin",
        "test_labels": folder / "test_labels.bin",
        "test_hashes": folder / "test_hashes.bin",
    }

    handles = {
        name: open(path, "wb")
        for name, path in binary_paths.items()
    }

    stats = {
        "train_total": 0,
        "train_positive": 0,
        "train_negative": 0,
        "test_total": 0,
        "test_positive": 0,
        "test_negative": 0,
        "dropped_bad_length": 0,
        "dropped_bad_alphabet": 0,
        "ambiguous_rows_kept": 0,
        "ambiguous_bases_kept": 0,
        "train_fold_counts": {
            str(f): {"0": 0, "1": 0}
            for f in range(N_FOLDS)
        },
    }

    try:
        for split_name, sources in [
            ("train", TRAIN_SOURCES),
            ("test", TEST_SOURCES),
        ]:
            for path, fixed_label in sources:
                seq_col, label_col = inspect_columns(path)

                if fixed_label is None and label_col is None:
                    raise ValueError(
                        f"{path} requires a binary label column."
                    )

                usecols = (
                    [seq_col]
                    if fixed_label is not None
                    else [seq_col, label_col]
                )

                print(
                    f"[{split_name.upper()}] {Path(path).name} | "
                    f"sequence={seq_col!r} | label={label_col!r} | "
                    f"fixed_label={fixed_label}"
                )

                for chunk in pd.read_csv(
                    path,
                    usecols=usecols,
                    chunksize=100_000,
                ):
                    seqs = normalize_sequences(chunk[seq_col])

                    length_ok = seqs.str.len().eq(SEQUENCE_LENGTH)
                    stats["dropped_bad_length"] += int((~length_ok).sum())

                    seqs = seqs[length_ok]
                    chunk = chunk.loc[length_ok]

                    alphabet_ok = seqs.map(
                        lambda x: bool(ALLOWED_RE.fullmatch(x))
                    )
                    stats["dropped_bad_alphabet"] += int((~alphabet_ok).sum())

                    seqs = seqs[alphabet_ok]
                    chunk = chunk.loc[alphabet_ok]

                    if len(seqs) == 0:
                        continue

                    amb = seqs.map(
                        lambda x: sum(ch not in "ACGT" for ch in x)
                    ).to_numpy(dtype=np.int16)

                    stats["ambiguous_rows_kept"] += int((amb > 0).sum())
                    stats["ambiguous_bases_kept"] += int(amb.sum())

                    if fixed_label is None:
                        y = parse_binary_labels(chunk[label_col])
                    else:
                        y = np.full(
                            len(seqs), fixed_label, dtype=np.uint8
                        )

                    codes = encode_sequence_chunk(seqs)

                    hashes = pd.util.hash_pandas_object(
                        seqs, index=False
                    ).to_numpy(dtype=np.uint64)

                    if split_name == "train":
                        folds = (
                            hashes % np.uint64(N_FOLDS)
                        ).astype(np.uint8)

                        append_binary(handles["train_codes"], codes)
                        append_binary(handles["train_labels"], y)
                        append_binary(handles["train_folds"], folds)
                        append_binary(handles["train_hashes"], hashes)

                        stats["train_total"] += len(y)
                        stats["train_positive"] += int(y.sum())
                        stats["train_negative"] += int(len(y) - y.sum())

                        for f in range(N_FOLDS):
                            m = folds == f
                            if not m.any():
                                continue
                            yf = y[m]
                            stats["train_fold_counts"][str(f)]["1"] += int(yf.sum())
                            stats["train_fold_counts"][str(f)]["0"] += int(
                                len(yf) - yf.sum()
                            )
                    else:
                        append_binary(handles["test_codes"], codes)
                        append_binary(handles["test_labels"], y)
                        append_binary(handles["test_hashes"], hashes)

                        stats["test_total"] += len(y)
                        stats["test_positive"] += int(y.sum())
                        stats["test_negative"] += int(len(y) - y.sum())

                    del seqs, chunk, y, codes, hashes
                    gc.collect()

    finally:
        for h in handles.values():
            h.close()

    manifest = {
        "source_signature": source_signature(),
        "stats": stats,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    save_json(manifest, folder / "manifest.json")
    return manifest

if not FORCE_REBUILD_PREPARED and cache_valid(LOCAL_PREP_DIR):
    print("[CACHE] Using existing local prepared data.")

elif (
    not FORCE_REBUILD_PREPARED
    and DRIVE_PREP_DIR.exists()
    and cache_valid(DRIVE_PREP_DIR)
):
    print("[CACHE] Copying prepared cache from Drive to local disk...")

    if LOCAL_PREP_DIR.exists():
        shutil.rmtree(LOCAL_PREP_DIR)

    shutil.copytree(DRIVE_PREP_DIR, LOCAL_PREP_DIR)
    print("[CACHE] Copy complete.")

else:
    print("[PREP] Building compact local cache...")
    build_prepared(LOCAL_PREP_DIR)

    if CACHE_PREPARED_TO_DRIVE:
        print("[CACHE] Saving prepared cache to Drive...")

        if DRIVE_PREP_DIR.exists():
            shutil.rmtree(DRIVE_PREP_DIR)

        shutil.copytree(LOCAL_PREP_DIR, DRIVE_PREP_DIR)
        print("[CACHE] Saved.")

MANIFEST = load_json(LOCAL_PREP_DIR / "manifest.json")
STATS = MANIFEST["stats"]

print(json.dumps(STATS, indent=2))

[PREP] Building compact local cache...
[TRAIN] train_positive_data.csv | sequence='Sequence' | label=None | fixed_label=1
[TRAIN] train_negative_data.csv | sequence='Sequence' | label=None | fixed_label=0
[TEST] test_positive_data.csv | sequence='Sequence' | label=None | fixed_label=1
[TEST] test_negative_data.csv | sequence='Sequence' | label=None | fixed_label=0
[CACHE] Saving prepared cache to Drive...
[CACHE] Saved.
{
  "train_total": 714661,
  "train_positive": 55800,
  "train_negative": 658861,
  "test_total": 178665,
  "test_positive": 13950,
  "test_negative": 164715,
  "dropped_bad_length": 0,
  "dropped_bad_alphabet": 0,
  "ambiguous_rows_kept": 5,
  "ambiguous_bases_kept": 34,
  "train_fold_counts": {
    "0": {
      "0": 132102,
      "1": 11160
    },
    "1": {
      "0": 131137,
      "1": 11003
    },
    "2": {
      "0": 131612,
      "1": 11308
    },
    "3": {
      "0": 132051,
      "1": 11213
    },
    "4": {
      "0": 131959,
      "1": 11116
    }
  }
}


In [ ]:
# ============================================================
# CELL 5 — DATASET IDENTITY + EXACT OVERLAP AUDIT
# ============================================================

N_TRAIN = int(STATS["train_total"])
N_TEST = int(STATS["test_total"])

identity = {
    "dataset_mode": DATASET_MODE,
    "development_total": N_TRAIN,
    "development_positive": int(STATS["train_positive"]),
    "development_negative": int(STATS["train_negative"]),
    "development_positive_rate": float(
        STATS["train_positive"] / max(1, N_TRAIN)
    ),
    "test_total": N_TEST,
    "test_positive": int(STATS["test_positive"]),
    "test_negative": int(STATS["test_negative"]),
}

save_json(identity, RESULT_DIR / "dataset_identity.json")

print(json.dumps(identity, indent=2))

if (
    identity["development_positive"] == 55_800
    and identity["development_negative"] == 658_861
):
    warnings.warn(
        "These PRIMARY development counts match the earlier SCLC-only split. "
        "Do not report this run as Combined-I (SCLC+NSCLC) unless the source "
        "files themselves are corrected."
    )

if RUN_EXACT_TRAIN_TEST_OVERLAP_AUDIT:
    train_hash = np.memmap(
        LOCAL_PREP_DIR / "train_hashes.bin",
        dtype=np.uint64,
        mode="r",
        shape=(N_TRAIN,),
    )

    test_hash = np.memmap(
        LOCAL_PREP_DIR / "test_hashes.bin",
        dtype=np.uint64,
        mode="r",
        shape=(N_TEST,),
    )

    print("\nRunning exact development/test sequence-overlap audit...")

    u_train = np.unique(np.asarray(train_hash))
    u_test = np.unique(np.asarray(test_hash))

    overlap = np.intersect1d(
        u_train, u_test, assume_unique=True
    )

    overlap_report = {
        "unique_development_sequences": int(len(u_train)),
        "unique_test_sequences": int(len(u_test)),
        "exact_hash_overlap_count": int(len(overlap)),
    }

    save_json(
        overlap_report,
        RESULT_DIR / "exact_overlap_audit.json"
    )

    print(json.dumps(overlap_report, indent=2))

    if len(overlap) > 0:
        warnings.warn(
            "Exact development/test sequence overlap exists. "
            "Investigate this before calling the test strictly leakage-free."
        )

    del train_hash, test_hash, u_train, u_test, overlap
    gc.collect()

{
  "dataset_mode": "PRIMARY",
  "development_total": 714661,
  "development_positive": 55800,
  "development_negative": 658861,
  "development_positive_rate": 0.07807897730532379,
  "test_total": 178665,
  "test_positive": 13950,
  "test_negative": 164715
}

Running exact development/test sequence-overlap audit...
{
  "unique_development_sequences": 714661,
  "unique_test_sequences": 178665,
  "exact_hash_overlap_count": 0
}


/tmp/ipykernel_2072/352927784.py:29: UserWarning: These PRIMARY development counts match the earlier SCLC-only split. Do not report this run as Combined-I (SCLC+NSCLC) unless the source files themselves are corrected.
  warnings.warn(


In [ ]:
# ============================================================
# CELL 6 — MEMMAP + ONE-HOT/NCP + FIXED-SHAPE PyDataset
# ============================================================

# Exact One-Hot + NCP mapping:
# A -> [1,0,0,0, 1,0,1]
# C -> [0,1,0,0, 0,1,1]
# G -> [0,0,1,0, 1,1,0]
# T -> [0,0,0,1, 0,0,0]
# IUPAC ambiguity -> all zeros.

ENCODING_LUT = np.zeros((5, 7), dtype=np.float32)

ENCODING_LUT[0] = [1,0,0,0, 1,0,1]
ENCODING_LUT[1] = [0,1,0,0, 0,1,1]
ENCODING_LUT[2] = [0,0,1,0, 1,1,0]
ENCODING_LUT[3] = [0,0,0,1, 0,0,0]

TRAIN_CODES = np.memmap(
    LOCAL_PREP_DIR / "train_codes.bin",
    dtype=np.uint8,
    mode="r",
    shape=(N_TRAIN, SEQUENCE_LENGTH),
)

TRAIN_LABELS = np.memmap(
    LOCAL_PREP_DIR / "train_labels.bin",
    dtype=np.uint8,
    mode="r",
    shape=(N_TRAIN,),
)

TRAIN_FOLDS = np.memmap(
    LOCAL_PREP_DIR / "train_folds.bin",
    dtype=np.uint8,
    mode="r",
    shape=(N_TRAIN,),
)

TEST_CODES = np.memmap(
    LOCAL_PREP_DIR / "test_codes.bin",
    dtype=np.uint8,
    mode="r",
    shape=(N_TEST, SEQUENCE_LENGTH),
)

TEST_LABELS = np.memmap(
    LOCAL_PREP_DIR / "test_labels.bin",
    dtype=np.uint8,
    mode="r",
    shape=(N_TEST,),
)

def encode_codes(codes):
    return ENCODING_LUT[
        np.asarray(codes, dtype=np.uint8)
    ].astype(np.float32, copy=False)

class NaturalTrainDataset(keras.utils.PyDataset):
    """
    Natural-distribution training dataset.
    Every epoch reshuffles the real training examples.
    No positive oversampling, no class weighting.
    """

    def __init__(
        self,
        codes,
        labels,
        indices,
        batch_size,
        seed,
    ):
        super().__init__()

        self.codes = codes
        self.labels = labels

        self.base_indices = np.asarray(
            indices, dtype=np.int64
        )

        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.epoch = 0

        # Fixed TPU shapes: drop only the small final remainder.
        self.steps = max(
            1,
            len(self.base_indices) // self.batch_size
        )

        self.usable = self.steps * self.batch_size
        self._shuffle()

    def _shuffle(self):
        rng = np.random.default_rng(
            self.seed + self.epoch
        )

        self.indices = self.base_indices.copy()
        rng.shuffle(self.indices)

        self.indices = self.indices[:self.usable]

    def __len__(self):
        return self.steps

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = start + self.batch_size

        sample_idx = self.indices[start:end]

        x = encode_codes(
            self.codes[sample_idx]
        )

        y = np.asarray(
            self.labels[sample_idx],
            dtype=np.float32
        )

        return x, y

    def on_epoch_end(self):
        self.epoch += 1
        self._shuffle()

class FixedEvalDataset(keras.utils.PyDataset):
    """
    Fixed-shape evaluation/prediction dataset for TPU/JAX.
    Last batch is padded; sample weights make padding invisible to metrics/loss.
    """

    def __init__(
        self,
        codes,
        labels,
        indices,
        batch_size,
        return_labels=True,
    ):
        super().__init__()

        self.codes = codes
        self.labels = labels
        self.indices = np.asarray(indices, dtype=np.int64)
        self.batch_size = int(batch_size)
        self.return_labels = bool(return_labels)

        self.steps = int(
            math.ceil(len(self.indices) / self.batch_size)
        )

    def __len__(self):
        return self.steps

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(
            start + self.batch_size,
            len(self.indices)
        )

        real_idx = self.indices[start:end]
        real_n = len(real_idx)

        if real_n == 0:
            raise IndexError(idx)

        if real_n < self.batch_size:
            pad_n = self.batch_size - real_n
            pad_idx = np.repeat(real_idx[:1], pad_n)
            batch_idx = np.concatenate([real_idx, pad_idx])
        else:
            pad_n = 0
            batch_idx = real_idx

        x = encode_codes(
            self.codes[batch_idx]
        )

        if not self.return_labels:
            return x

        y = np.asarray(
            self.labels[batch_idx],
            dtype=np.float32
        )

        sw = np.ones(
            self.batch_size,
            dtype=np.float32
        )

        if pad_n:
            sw[-pad_n:] = 0.0

        return x, y, sw

print("Natural-distribution data pipeline ready.")

Natural-distribution data pipeline ready.


In [ ]:
# ============================================================
# CELL 7 — HIGH-CAPACITY CGRU-5mC MODEL
# ============================================================

@keras.utils.register_keras_serializable(package="CGRU5mC")
class CBAM1D(layers.Layer):
    def __init__(
        self,
        reduction=8,
        spatial_kernel=7,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.reduction = int(reduction)
        self.spatial_kernel = int(spatial_kernel)

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden = max(channels // self.reduction, 8)

        self.mlp1 = layers.Dense(
            hidden,
            activation="relu",
            kernel_initializer="he_normal",
        )

        self.mlp2 = layers.Dense(
            channels,
            kernel_initializer="he_normal",
        )

        self.spatial_conv = layers.Conv1D(
            1,
            self.spatial_kernel,
            padding="same",
            activation="sigmoid",
        )

        super().build(input_shape)

    def call(self, x):
        avg = ops.mean(x, axis=1)
        mx = ops.max(x, axis=1)

        ca = ops.sigmoid(
            self.mlp2(self.mlp1(avg))
            + self.mlp2(self.mlp1(mx))
        )

        x = x * ops.expand_dims(ca, axis=1)

        avg_spatial = ops.mean(
            x, axis=-1, keepdims=True
        )

        max_spatial = ops.max(
            x, axis=-1, keepdims=True
        )

        sa = self.spatial_conv(
            ops.concatenate(
                [avg_spatial, max_spatial],
                axis=-1
            )
        )

        return x * sa

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "reduction": self.reduction,
                "spatial_kernel": self.spatial_kernel,
            }
        )
        return config

def conv_bn_relu(
    x,
    filters,
    kernel_size,
    dilation=1,
    name=None,
):
    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="same",
        dilation_rate=dilation,
        use_bias=False,
        kernel_initializer="he_normal",
        name=None if name is None else f"{name}_conv",
    )(x)

    x = layers.BatchNormalization(
        name=None if name is None else f"{name}_bn"
    )(x)

    return layers.Activation(
        "relu",
        name=None if name is None else f"{name}_relu"
    )(x)

def build_model():
    inp = layers.Input(
        shape=(SEQUENCE_LENGTH, 7),
        dtype="float32",
        name="onehot_ncp_input",
    )

    # 1) Multi-scale motif extraction
    b3 = conv_bn_relu(
        inp, BRANCH_FILTERS, 3, name="ms_k3"
    )
    b5 = conv_bn_relu(
        inp, BRANCH_FILTERS, 5, name="ms_k5"
    )
    b7 = conv_bn_relu(
        inp, BRANCH_FILTERS, 7, name="ms_k7"
    )

    x = layers.Concatenate(
        name="multiscale_concat"
    )([b3, b5, b7])

    x = layers.SpatialDropout1D(
        SPATIAL_DROPOUT,
        name="spatial_dropout"
    )(x)

    # 2) Dilated residual block
    channels = BRANCH_FILTERS * 3
    residual = x
    d = x

    for rate in DILATION_RATES:
        d = conv_bn_relu(
            d,
            channels,
            3,
            dilation=rate,
            name=f"dilated_{rate}",
        )

    d = layers.Conv1D(
        channels,
        1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="dilated_projection",
    )(d)

    d = layers.BatchNormalization(
        name="dilated_projection_bn"
    )(d)

    x = layers.Add(
        name="dilated_residual_add"
    )([residual, d])

    x = layers.Activation(
        "relu",
        name="dilated_residual_relu"
    )(x)

    # 3) CBAM
    x = CBAM1D(
        reduction=CBAM_REDUCTION,
        spatial_kernel=7,
        name="cbam",
    )(x)

    # 4) Larger BiGRU
    x = layers.Bidirectional(
        layers.GRU(
            BIGRU_UNITS,
            return_sequences=True,
            dropout=0.0,
            recurrent_dropout=0.0,
            reset_after=True,
        ),
        name="bigru",
    )(x)

    # 5) Multi-head self-attention
    attn = layers.MultiHeadAttention(
        num_heads=MHA_HEADS,
        key_dim=MHA_KEY_DIM,
        dropout=ATTENTION_DROPOUT,
        name="mha",
    )(x, x)

    attn = layers.Dropout(
        ATTENTION_DROPOUT,
        name="mha_dropout"
    )(attn)

    x = layers.Add(
        name="mha_residual"
    )([x, attn])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name="mha_norm"
    )(x)

    # 6) Larger FFN
    feature_dim = int(x.shape[-1])

    ff = layers.Dense(
        FFN_UNITS,
        activation="relu",
        kernel_initializer="he_normal",
        name="ffn_dense1",
    )(x)

    ff = layers.Dropout(
        DROPOUT,
        name="ffn_dropout"
    )(ff)

    ff = layers.Dense(
        feature_dim,
        name="ffn_dense2"
    )(ff)

    x = layers.Add(
        name="ffn_residual"
    )([x, ff])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name="ffn_norm"
    )(x)

    # 7) Dual global pooling
    avg_pool = layers.GlobalAveragePooling1D(
        name="global_average_pool"
    )(x)

    if USE_DUAL_GLOBAL_POOLING:
        max_pool = layers.GlobalMaxPooling1D(
            name="global_max_pool"
        )(x)

        x = layers.Concatenate(
            name="dual_global_pool"
        )([avg_pool, max_pool])
    else:
        x = avg_pool

    # 8) Larger dense classifier
    x = layers.Dense(
        DENSE_1,
        activation="relu",
        kernel_initializer="he_normal",
        name="dense_1",
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="dense_1_dropout"
    )(x)

    x = layers.Dense(
        DENSE_2,
        activation="relu",
        kernel_initializer="he_normal",
        name="dense_2",
    )(x)

    x = layers.Dropout(
        DROPOUT * 0.5,
        name="dense_2_dropout"
    )(x)

    out = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="probability",
    )(x)

    return keras.Model(
        inp,
        out,
        name="CGRU5mC_V9_Fold97_FullDev"
    )

def compile_model(model):
    try:
        optimizer = keras.optimizers.AdamW(
            learning_rate=INITIAL_LR,
            weight_decay=WEIGHT_DECAY,
            clipnorm=1.0,
            use_ema=True,
            ema_momentum=0.999,
        )
    except TypeError:
        optimizer = keras.optimizers.AdamW(
            learning_rate=INITIAL_LR,
            weight_decay=WEIGHT_DECAY,
            clipnorm=1.0,
        )

    model.compile(
        optimizer=optimizer,

        loss=keras.losses.BinaryCrossentropy(
            label_smoothing=LABEL_SMOOTHING
        ),

        metrics=[
            keras.metrics.BinaryAccuracy(
                name="accuracy",
                threshold=FIXED_THRESHOLD,
            ),
            keras.metrics.Precision(
                name="precision",
                thresholds=FIXED_THRESHOLD,
            ),
            keras.metrics.Recall(
                name="sensitivity",
                thresholds=FIXED_THRESHOLD,
            ),
            keras.metrics.AUC(
                name="auc",
                curve="ROC",
                num_thresholds=500,
            ),
            keras.metrics.AUC(
                name="pr_auc",
                curve="PR",
                num_thresholds=500,
            ),
        ],

        jit_compile=True,
    )

    return model

def fresh_model(seed):
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)

    model = build_model()
    return compile_model(model)

probe = fresh_model(SEED)
probe.summary()

del probe
keras.backend.clear_session()
gc.collect()

print("V9 fold-97/full-development model ready.")

Model: "CGRU5mC_V9_Fold97_FullDev"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ onehot_ncp_input    │ (None, 41, 7)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k3_conv (Conv1D) │ (None, 41, 88)    │      1,848 │ onehot_ncp_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k5_conv (Conv1D) │ (None, 41, 88)    │      3,080 │ onehot_ncp_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k7_conv (Conv1D) │ (None, 41, 88)    │      4,312 │ onehot_ncp_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k3_bn            │ (None, 41, 88)    │        352 │ ms_k3_conv[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k5_bn            │ (None, 41, 88)    │        352 │ ms_k5_conv[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k7_bn            │ (None, 41, 88)    │        352 │ ms_k7_conv[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k3_relu          │ (None, 41, 88)    │          0 │ ms_k3_bn[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k5_relu          │ (None, 41, 88)    │          0 │ ms_k5_bn[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ms_k7_relu          │ (None, 41, 88)    │          0 │ ms_k7_bn[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiscale_concat   │ (None, 41, 264)   │          0 │ ms_k3_relu[0][0], │
│ (Concatenate)       │                   │            │ ms_k5_relu[0][0], │
│                     │                   │            │ ms_k7_relu[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 41, 264)   │          0 │ multiscale_conca… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dilated_1_conv      │ (None, 41, 264)   │    209,088 │ spatial_dropout[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dilated_1_bn        │ (None, 41, 264)   │      1,056 │ dilated_1_conv[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dilated_1_relu      │ (None, 41, 264)   │          0 │ dilated_1_bn[0][… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dilated_2_conv      │ (None, 41, 264)   │    209,088 │ dilated_1_relu[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dilated_2_bn        │ (None, 41, 264)   │      1,056 │ dilated_2_conv[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,863,073 (7.11 MB)

 Trainable params: 1,860,433 (7.10 MB)

 Non-trainable params: 2,640 (10.31 KB)

V9 fold-97/full-development model ready.


In [ ]:
# ============================================================
# CELL 8 — EXACT METRICS + PLOTS @ FIXED THRESHOLD 0.50
# ============================================================

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(
        y_true, dtype=np.uint8
    )

    y_prob = np.asarray(
        y_prob, dtype=np.float32
    ).reshape(-1)

    y_pred = (
        y_prob >= FIXED_THRESHOLD
    ).astype(np.uint8)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "threshold": float(FIXED_THRESHOLD),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            precision_score(
                y_true, y_pred, zero_division=0
            )
        ),
        "sensitivity": float(
            tp / (tp + fn) if (tp + fn) else 0.0
        ),
        "specificity": float(
            tn / (tn + fp) if (tn + fp) else 0.0
        ),
        "mcc": float(
            matthews_corrcoef(y_true, y_pred)
        ),
        "f1": float(
            f1_score(
                y_true, y_pred, zero_division=0
            )
        ),
        "auc": float(
            roc_auc_score(y_true, y_prob)
        ),
        "pr_auc": float(
            average_precision_score(
                y_true, y_prob
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def save_confusion_matrix(
    y_true,
    y_prob,
    path,
    title,
):
    pred = (
        np.asarray(y_prob)
        >= FIXED_THRESHOLD
    ).astype(np.uint8)

    cm = confusion_matrix(
        y_true, pred, labels=[0, 1]
    )

    fig, ax = plt.subplots(
        figsize=(6.2, 5.2)
    )

    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(
        [0, 1],
        ["Non-5mC", "5mC"]
    )
    ax.set_yticks(
        [0, 1],
        ["Non-5mC", "5mC"]
    )

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center"
            )

    fig.colorbar(im, ax=ax)
    fig.tight_layout()

    fig.savefig(
        path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)

def save_roc_pr(
    y_true,
    y_prob,
    prefix,
):
    fpr, tpr, _ = roc_curve(
        y_true, y_prob
    )

    auc = roc_auc_score(
        y_true, y_prob
    )

    fig, ax = plt.subplots(
        figsize=(6.4, 5.4)
    )

    ax.plot(
        fpr, tpr,
        label=f"AUC = {auc:.4f}"
    )
    ax.plot([0,1],[0,1],"--")

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)

    fig.tight_layout()

    fig.savefig(
        RESULT_DIR / f"{prefix}_roc.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)

    precision, recall, _ = precision_recall_curve(
        y_true, y_prob
    )

    ap = average_precision_score(
        y_true, y_prob
    )

    fig, ax = plt.subplots(
        figsize=(6.4, 5.4)
    )

    ax.plot(
        recall, precision,
        label=f"AP = {ap:.4f}"
    )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.legend(loc="lower left")
    ax.grid(alpha=0.25)

    fig.tight_layout()

    fig.savefig(
        RESULT_DIR / f"{prefix}_pr.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)

print("Fixed-threshold metric helpers ready.")

Fixed-threshold metric helpers ready.


In [ ]:
# ============================================================
# CELL 9A — GENERALIZATION MONITOR
# ============================================================

class GeneralizationMonitor(keras.callbacks.Callback):
    """
    Prints the train/validation accuracy gap after each epoch.
    It does not change training; it only makes overfitting visible.
    """

    def __init__(self, target_low=0.96, target_high=0.97):
        super().__init__()
        self.target_low = float(target_low)
        self.target_high = float(target_high)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        train_acc = logs.get("accuracy")
        val_acc = logs.get("val_accuracy")

        if train_acc is None or val_acc is None:
            return

        gap = float(train_acc - val_acc)

        msg = (
            f"[V8 monitor] epoch={epoch + 1} | "
            f"train_acc={train_acc:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"gap={gap:.4f}"
        )

        if self.target_low <= train_acc <= self.target_high:
            msg += " | train accuracy is inside the ~96–97% target band"

        if gap >= 0.035:
            msg += " | warning: train–validation gap is becoming large"

        print(msg)

In [ ]:
# ============================================================
# V9 — FULL-DEVELOPMENT TRAINING HELPERS
# ============================================================

class StopAtTargetTrainingAccuracy(keras.callbacks.Callback):
    """
    Stops a final full-development model after it reaches the predeclared
    training-accuracy target, but not before FINAL_MIN_EPOCHS.
    This callback never reads validation or independent-test performance.
    """

    def __init__(self, target, min_epochs):
        super().__init__()
        self.target = float(target)
        self.min_epochs = int(min_epochs)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_acc = logs.get("accuracy")

        if train_acc is None:
            return

        completed_epoch = epoch + 1

        print(
            f"[full-dev monitor] epoch={completed_epoch} | "
            f"train_acc={float(train_acc):.4f} | "
            # f"target={self.target:.4f}"
        )

        if (
            completed_epoch >= self.min_epochs
            and float(train_acc) >= self.target
        ):
            print(
                # f"Reached final-development target accuracy "
                f"{self.target:.4f}; stopping."
            )
            self.model.stop_training = True


def build_full_development_dataset(seed):
    all_indices = np.arange(
        N_TRAIN,
        dtype=np.int64
    )

    return NaturalTrainDataset(
        codes=TRAIN_CODES,
        labels=TRAIN_LABELS,
        indices=all_indices,
        batch_size=TRAIN_BATCH_SIZE,
        seed=seed,
    )


def cv_derived_final_epoch_cap(fold_results):
    """
    The final epoch cap is derived only from CV behavior.
    It deliberately allows several epochs beyond the median best-validation
    epoch so the model trained on 100% of development data can fit more fully,
    but it is capped to avoid V6-style extreme memorization.
    """
    best_val_epochs = [
        int(r["max_validation_accuracy_epoch"])
        for r in fold_results
    ]

    median_best_val_epoch = int(
        round(np.median(best_val_epochs))
    )

    derived = median_best_val_epoch + 9

    return int(
        min(
            FINAL_MAX_EPOCHS,
            max(
                FINAL_MIN_EPOCHS,
                derived
            )
        )
    )

print("V9 full-development helpers ready.")

V9 full-development helpers ready.


In [ ]:
# ============================================================
# CELL 9 — TRAIN ONE FOLD
# ============================================================
#
# TWO checkpoints are saved:
# 1) best_train.weights.h5 -> highest TRAINING accuracy
# 2) best_val.weights.h5   -> highest VALIDATION accuracy
#
# OOF/test uses best_val so the independent evaluation remains meaningful.
# ============================================================

def predict_indices(
    model,
    codes,
    labels,
    indices,
):
    ds = FixedEvalDataset(
        codes=codes,
        labels=labels,
        indices=indices,
        batch_size=EVAL_BATCH_SIZE,
        return_labels=False,
    )

    raw = np.asarray(
        model.predict(
            ds,
            verbose=1
        )
    ).reshape(-1)

    # model.predict includes padded rows in the final fixed-size batch.
    return raw[:len(indices)].astype(np.float32)

def train_one_fold(fold):
    fold_dir = RESULT_DIR / f"fold_{fold + 1}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    done_file = fold_dir / "fold_complete.json"
    val_prob_file = fold_dir / "val_prob.npy"
    val_idx_file = fold_dir / "val_indices.npy"

    best_train_weights = fold_dir / "best_train.weights.h5"
    best_val_weights = fold_dir / "best_val.weights.h5"

    if (
        done_file.exists()
        and val_prob_file.exists()
        and val_idx_file.exists()
        and best_train_weights.exists()
        and best_val_weights.exists()
    ):
        print(f"[FOLD {fold + 1}] Already complete -> skipping.")
        return load_json(done_file)

    folds_array = np.asarray(TRAIN_FOLDS)

    train_indices = np.flatnonzero(
        folds_array != fold
    )

    val_indices = np.flatnonzero(
        folds_array == fold
    )

    train_ds = NaturalTrainDataset(
        codes=TRAIN_CODES,
        labels=TRAIN_LABELS,
        indices=train_indices,
        batch_size=TRAIN_BATCH_SIZE,
        seed=SEED + fold,
    )

    val_ds = FixedEvalDataset(
        codes=TRAIN_CODES,
        labels=TRAIN_LABELS,
        indices=val_indices,
        batch_size=EVAL_BATCH_SIZE,
        return_labels=True,
    )

    print("\n" + "=" * 80)
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print("Natural train samples:", len(train_indices))
    print("Train batches/epoch:", len(train_ds))
    print("Validation samples:", len(val_indices))
    print("Threshold:", FIXED_THRESHOLD)
    print("=" * 80)

    model = fresh_model(
        SEED + fold
    )

    callbacks = [
        GeneralizationMonitor(target_low=0.96, target_high=0.97),
        # TRAIN-ACCURACY checkpoint
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_train_weights),
            monitor="accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),

        # VALIDATION checkpoint, kept separately
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_val_weights),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),

        # Validation-driven LR reduction helps generalization instead of memorization.
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy",
            mode="max",
            factor=0.5,
            patience=LR_PATIENCE,
            min_lr=MIN_LR,
            verbose=1,
        ),

        keras.callbacks.CSVLogger(
            str(fold_dir / "training_log.csv"),
            append=False,
        ),

        keras.callbacks.TerminateOnNaN(),
    ]
    callbacks.append(
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=FOLD_EARLY_STOP_PATIENCE,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        )
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    max_train_acc = float(
        np.max(history.history["accuracy"])
    )

    max_train_epoch = int(
        np.argmax(history.history["accuracy"]) + 1
    )

    max_val_acc = float(
        np.max(history.history["val_accuracy"])
    )

    max_val_epoch = int(
        np.argmax(history.history["val_accuracy"]) + 1
    )

    # OOF predictions use the best VALIDATION checkpoint.
    model.load_weights(best_val_weights)

    val_prob = predict_indices(
        model=model,
        codes=TRAIN_CODES,
        labels=TRAIN_LABELS,
        indices=val_indices,
    )

    np.save(
        val_prob_file,
        val_prob
    )

    np.save(
        val_idx_file,
        val_indices
    )

    val_y = np.asarray(
        TRAIN_LABELS[val_indices],
        dtype=np.uint8
    )

    val_metrics = compute_metrics(
        val_y,
        val_prob
    )

    result = {
        "fold": fold + 1,
        "max_training_accuracy": max_train_acc,
        "max_training_accuracy_epoch": max_train_epoch,
        "max_validation_accuracy": max_val_acc,
        "max_validation_accuracy_epoch": max_val_epoch,
        "validation_metrics_at_fixed_0_5": val_metrics,
    }

    save_json(
        result,
        done_file
    )

    print("\nFOLD SUMMARY")
    print(json.dumps(result, indent=2))

    del (
        model,
        train_ds,
        val_ds,
        train_indices,
        val_indices,
        val_prob,
        val_y,
        history,
    )

    keras.backend.clear_session()
    gc.collect()

    return result

print("Fold trainer ready.")

Fold trainer ready.


In [ ]:
# ============================================================
# CELL 10 — RUN FIVE-FOLD TRAINING
# ============================================================

fold_results = []

for fold in range(N_FOLDS):
    fold_results.append(
        train_one_fold(fold)
    )

fold_summary = pd.DataFrame(
    [
        {
            "Fold": r["fold"],
            "Max_Training_Accuracy": r["max_training_accuracy"],
            "Train_Best_Epoch": r["max_training_accuracy_epoch"],
            "Max_Validation_Accuracy": r["max_validation_accuracy"],
            "Validation_Best_Epoch": r["max_validation_accuracy_epoch"],
            "Train_Val_Gap": r["max_training_accuracy"] - r["max_validation_accuracy"],
        }
        for r in fold_results
    ]
)

fold_summary.to_csv(
    RESULT_DIR / "training_accuracy_summary.csv",
    index=False,
)

display(fold_summary)

print("\nMean max TRAINING accuracy:",
      fold_summary["Max_Training_Accuracy"].mean())

print("Highest fold TRAINING accuracy:",
      fold_summary["Max_Training_Accuracy"].max())

print("Mean max VALIDATION accuracy:",
      fold_summary["Max_Validation_Accuracy"].mean())

print("Mean train-validation gap:",
      fold_summary["Train_Val_Gap"].mean())


FOLD 1/5
Natural train samples: 571399
Train batches/epoch: 1116
Validation samples: 143262
Threshold: 0.5
Epoch 1/55
1112/1116 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9164 - auc: 0.8685 - loss: 0.2131 - pr_auc: 0.3348 - precision: 0.3544 - sensitivity: 0.1160[V8 monitor] epoch=1 | train_acc=0.9249 | val_acc=0.9336 | gap=-0.0088

Epoch 1: accuracy improved from None to 0.92487, saving model to /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY/fold_1/best_train.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY/fold_1/best_train.weights.h5

Epoch 1: val_accuracy improved from None to 0.93362, saving model to /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY/fold_1/best_val.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY/fold_1/best_val.weights.h5
1116/1116 ━━━━━━━━━━━━━━━━━━━━ 74s 17ms/step - accuracy: 0.9249 - a

,Fold,Max_Training_Accuracy,Train_Best_Epoch,Max_Validation_Accuracy,Validation_Best_Epoch,Train_Val_Gap
0,1,0.953895,20,0.936140,6,0.017755
1,2,0.956713,23,0.937640,9,0.019074
2,3,0.961732,21,0.935812,7,0.025920
3,4,0.952085,20,0.935833,6,0.016253
4,5,0.957183,22,0.936579,8,0.020604



Mean max TRAINING accuracy: 0.9563218474388122
Highest fold TRAINING accuracy: 0.961732029914856
Mean max VALIDATION accuracy: 0.9364006757736206
Mean train-validation gap: 0.01992117166519165


In [ ]:
# ============================================================
# CELL 11 — OOF RESULTS @ FIXED THRESHOLD 0.50
# ============================================================

oof_prob = np.full(
    N_TRAIN,
    np.nan,
    dtype=np.float32
)

for fold in range(N_FOLDS):
    fold_dir = RESULT_DIR / f"fold_{fold + 1}"

    idx = np.load(
        fold_dir / "val_indices.npy"
    )

    prob = np.load(
        fold_dir / "val_prob.npy"
    )

    if len(idx) != len(prob):
        raise RuntimeError(
            f"Fold {fold + 1} OOF length mismatch."
        )

    oof_prob[idx] = prob

if np.isnan(oof_prob).any():
    raise RuntimeError("Incomplete OOF predictions.")

oof_y = np.asarray(
    TRAIN_LABELS,
    dtype=np.uint8
)

oof_metrics = compute_metrics(
    oof_y,
    oof_prob
)

np.save(
    RESULT_DIR / "oof_y_true.npy",
    oof_y
)

np.save(
    RESULT_DIR / "oof_prob.npy",
    oof_prob
)

save_json(
    {
        "threshold": FIXED_THRESHOLD,
        "threshold_tuning": False,
        "oof_metrics": oof_metrics,
    },
    RESULT_DIR / "oof_summary.json",
)

save_confusion_matrix(
    oof_y,
    oof_prob,
    RESULT_DIR / "oof_confusion_matrix.png",
    f"OOF CV @ fixed threshold {FIXED_THRESHOLD:.2f}",
)

save_roc_pr(
    oof_y,
    oof_prob,
    "oof"
)

print("OOF METRICS @ FIXED 0.50")
print(json.dumps(oof_metrics, indent=2))

OOF METRICS @ FIXED 0.50
{
  "threshold": 0.5,
  "accuracy": 0.9362103151004463,
  "precision": 0.5803664179809236,
  "sensitivity": 0.6608064516129032,
  "specificity": 0.9595347121775306,
  "mcc": 0.5848017302903227,
  "f1": 0.6179797878224144,
  "auc": 0.9644441652888545,
  "pr_auc": 0.58849160869297,
  "tn": 632200,
  "fp": 26661,
  "fn": 18927,
  "tp": 36873
}


In [ ]:
# ============================================================
# CELL 12 — FINAL FULL-DEVELOPMENT ENSEMBLE + INDEPENDENT TEST
# ============================================================
#
# Two sources are combined:
#
# A) 5 best-validation CV models
#    -> strong validation-controlled generalization anchor.
#
# B) 3 models trained on ALL development samples
#    -> gives each final model access to 100% of the development data.
#
# The blend weights are fixed in Cell 1 and are NOT selected on the test set.
# Threshold remains exactly 0.50.
# ============================================================

test_indices = np.arange(
    N_TEST,
    dtype=np.int64
)

# ------------------------------------------------------------
# A) FIVE-FOLD BEST-VALIDATION ENSEMBLE
# ------------------------------------------------------------
cv_test_probabilities = []

for fold in range(N_FOLDS):
    fold_dir = RESULT_DIR / f"fold_{fold + 1}"
    weights_file = fold_dir / "best_val.weights.h5"

    if not weights_file.exists():
        raise FileNotFoundError(weights_file)

    print(
        f"\nCV ensemble test prediction: fold {fold + 1}/{N_FOLDS}"
    )

    model = fresh_model(SEED + fold)
    model.load_weights(weights_file)

    prob = predict_indices(
        model=model,
        codes=TEST_CODES,
        labels=TEST_LABELS,
        indices=test_indices,
    )

    cv_test_probabilities.append(prob)

    del model, prob
    keras.backend.clear_session()
    gc.collect()

cv_test_prob = np.mean(
    np.stack(
        cv_test_probabilities,
        axis=0
    ),
    axis=0,
).astype(np.float32)

np.save(
    RESULT_DIR / "test_prob_cv5_ensemble.npy",
    cv_test_prob
)

# ------------------------------------------------------------
# B) FULL-DEVELOPMENT ENSEMBLE
# ------------------------------------------------------------
FINAL_EPOCH_CAP = cv_derived_final_epoch_cap(
    fold_results
)

print("\n" + "=" * 80)
print("FINAL FULL-DEVELOPMENT TRAINING")
print("CV-derived epoch cap:", FINAL_EPOCH_CAP)
print("Seeds:", FINAL_ENSEMBLE_SEEDS)
print("=" * 80)

full_dev_test_probabilities = []
full_dev_training_summaries = []

for run_index, seed in enumerate(
    FINAL_ENSEMBLE_SEEDS,
    start=1
):
    final_dir = (
        RESULT_DIR
        / f"full_development_seed_{seed}"
    )

    final_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    weights_file = (
        final_dir
        / "final.weights.h5"
    )

    summary_file = (
        final_dir
        / "training_summary.json"
    )

    print(
        f"\nFULL DEVELOPMENT MODEL "
        f"{run_index}/{len(FINAL_ENSEMBLE_SEEDS)} | seed={seed}"
    )

    model = fresh_model(seed)

    full_train_ds = build_full_development_dataset(
        seed
    )

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(weights_file),
            monitor="accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="loss",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=MIN_LR,
            verbose=1,
        ),

        StopAtTargetTrainingAccuracy(
            target=FINAL_TARGET_TRAIN_ACCURACY,
            min_epochs=FINAL_MIN_EPOCHS,
        ),

        keras.callbacks.CSVLogger(
            str(
                final_dir
                / "training_log.csv"
            ),
            append=False,
        ),

        keras.callbacks.TerminateOnNaN(),
    ]

    history = model.fit(
        full_train_ds,
        epochs=FINAL_EPOCH_CAP,
        callbacks=callbacks,
        verbose=1,
    )

    max_full_train_acc = float(
        np.max(
            history.history["accuracy"]
        )
    )

    best_full_train_epoch = int(
        np.argmax(
            history.history["accuracy"]
        )
        + 1
    )

    model.load_weights(
        weights_file
    )

    prob = predict_indices(
        model=model,
        codes=TEST_CODES,
        labels=TEST_LABELS,
        indices=test_indices,
    )

    np.save(
        final_dir
        / "test_prob.npy",
        prob
    )

    full_summary = {
        "seed": int(seed),
        "epoch_cap_from_cv": int(FINAL_EPOCH_CAP),
        "max_training_accuracy": max_full_train_acc,
        "max_training_accuracy_epoch": best_full_train_epoch,
        "target_training_accuracy": float(

        ),
    }

    save_json(
        full_summary,
        summary_file
    )

    full_dev_training_summaries.append(
        full_summary
    )

    full_dev_test_probabilities.append(
        prob
    )

    print(
        json.dumps(
            full_summary,
            indent=2
        )
    )

    del (
        model,
        full_train_ds,
        history,
        prob,
    )

    keras.backend.clear_session()
    gc.collect()

full_dev_test_prob = np.mean(
    np.stack(
        full_dev_test_probabilities,
        axis=0
    ),
    axis=0,
).astype(np.float32)

np.save(
    RESULT_DIR
    / "test_prob_full_development_ensemble.npy",
    full_dev_test_prob
)

# ------------------------------------------------------------
# C) PREDECLARED BLEND
# ------------------------------------------------------------
if not np.isclose(
    FINAL_BLEND_CV_WEIGHT
    + FINAL_BLEND_FULLDEV_WEIGHT,
    1.0
):
    raise ValueError(
        "FINAL_BLEND_CV_WEIGHT + FINAL_BLEND_FULLDEV_WEIGHT must equal 1."
    )

test_prob = (
    FINAL_BLEND_CV_WEIGHT
    * cv_test_prob
    + FINAL_BLEND_FULLDEV_WEIGHT
    * full_dev_test_prob
).astype(np.float32)

y_test = np.asarray(
    TEST_LABELS,
    dtype=np.uint8
)

test_metrics = compute_metrics(
    y_test,
    test_prob
)

np.save(
    RESULT_DIR
    / "test_y_true.npy",
    y_test
)

np.save(
    RESULT_DIR
    / "test_prob_final_blend.npy",
    test_prob
)

pd.DataFrame(
    {
        "y_true": y_test,
        "y_prob": test_prob,
        "y_pred": (
            test_prob
            >= FIXED_THRESHOLD
        ).astype(np.uint8),
    }
).to_csv(
    RESULT_DIR
    / "test_predictions_final_blend.csv.gz",
    index=False,
    compression="gzip",
)

save_json(
    {
        "fixed_threshold": FIXED_THRESHOLD,
        "threshold_tuning": False,
        "cv_ensemble_size": N_FOLDS,
        "full_development_ensemble_size": len(
            FINAL_ENSEMBLE_SEEDS
        ),
        "cv_weight": FINAL_BLEND_CV_WEIGHT,
        "full_development_weight": FINAL_BLEND_FULLDEV_WEIGHT,
        "final_epoch_cap_from_cv": FINAL_EPOCH_CAP,
        "full_development_training": full_dev_training_summaries,
        "test_metrics": test_metrics,
    },
    RESULT_DIR
    / "test_metrics.json",
)

save_confusion_matrix(
    y_test,
    test_prob,
    RESULT_DIR
    / "test_confusion_matrix.png",
    f"Independent Test @ fixed threshold {FIXED_THRESHOLD:.2f}",
)

save_roc_pr(
    y_test,
    test_prob,
    "test"
)

print("\nFINAL INDEPENDENT TEST")
print("=" * 70)
print("CV ensemble weight:", FINAL_BLEND_CV_WEIGHT)
print("Full-development ensemble weight:", FINAL_BLEND_FULLDEV_WEIGHT)
print("Fixed threshold:", FIXED_THRESHOLD)
print(json.dumps(test_metrics, indent=2))


CV ensemble test prediction: fold 1/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step

CV ensemble test prediction: fold 2/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step

CV ensemble test prediction: fold 3/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step

CV ensemble test prediction: fold 4/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step

CV ensemble test prediction: fold 5/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step

FINAL FULL-DEVELOPMENT TRAINING
CV-derived epoch cap: 25
Seeds: (42, 52, 62)

FULL DEVELOPMENT MODEL 1/3 | seed=42
Epoch 1/25
1393/1395 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9183 - auc: 0.8849 - loss: 0.2006 - pr_auc: 0.3614 - precision: 0.3832 - sensitivity: 0.1516
Epoch 1: accuracy improved from None to 0.92638, saving model to /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY/full_development_seed_42/final.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/5Mc/CGRU5mC_V9_Fold97_FullDev_FIXED05/PRIMARY/full_development_seed_42/final.weights.h5
[full-dev 

In [ ]:
# ============================================================
# CELL 13 — FINAL SUMMARY
# ============================================================

training_mean = float(
    fold_summary["Max_Training_Accuracy"].mean()
)

training_max = float(
    fold_summary["Max_Training_Accuracy"].max()
)

validation_mean = float(
    fold_summary["Max_Validation_Accuracy"].mean()
)

full_dev_train_mean = float(
    np.mean(
        [
            x["max_training_accuracy"]
            for x in full_dev_training_summaries
        ]
    )
)

full_dev_train_max = float(
    np.max(
        [
            x["max_training_accuracy"]
            for x in full_dev_training_summaries
        ]
    )
)

summary_df = pd.DataFrame(
    [
        {
            "Evaluation": "5-Fold max training accuracy (mean)",
            "Threshold": FIXED_THRESHOLD,
            "Accuracy": training_mean,
            "Precision": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "MCC": np.nan,
            "F1": np.nan,
            "ROC_AUC": np.nan,
            "PR_AUC": np.nan,
        },
        {
            "Evaluation": "Highest single-fold training accuracy",
            "Threshold": FIXED_THRESHOLD,
            "Accuracy": training_max,
            "Precision": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "MCC": np.nan,
            "F1": np.nan,
            "ROC_AUC": np.nan,
            "PR_AUC": np.nan,
        },
        {
            "Evaluation": "Full-development ensemble training accuracy (mean)",
            "Threshold": FIXED_THRESHOLD,
            "Accuracy": full_dev_train_mean,
            "Precision": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "MCC": np.nan,
            "F1": np.nan,
            "ROC_AUC": np.nan,
            "PR_AUC": np.nan,
        },
        {
            "Evaluation": "OOF CV",
            "Threshold": FIXED_THRESHOLD,
            "Accuracy": oof_metrics["accuracy"],
            "Precision": oof_metrics["precision"],
            "Sensitivity": oof_metrics["sensitivity"],
            "Specificity": oof_metrics["specificity"],
            "MCC": oof_metrics["mcc"],
            "F1": oof_metrics["f1"],
            "ROC_AUC": oof_metrics["auc"],
            "PR_AUC": oof_metrics["pr_auc"],
        },
        {
            "Evaluation": "Independent Test — V9 blended ensemble",
            "Threshold": FIXED_THRESHOLD,
            "Accuracy": test_metrics["accuracy"],
            "Precision": test_metrics["precision"],
            "Sensitivity": test_metrics["sensitivity"],
            "Specificity": test_metrics["specificity"],
            "MCC": test_metrics["mcc"],
            "F1": test_metrics["f1"],
            "ROC_AUC": test_metrics["auc"],
            "PR_AUC": test_metrics["pr_auc"],
        },
    ]
)

summary_df.to_csv(
    RESULT_DIR
    / "final_summary.csv",
    index=False
)

save_json(
    {
        "version": "V9_Fold97_FullDev_FIXED05",
        "dataset_mode": DATASET_MODE,
        "fixed_threshold": FIXED_THRESHOLD,
        "mean_max_fold_training_accuracy": training_mean,
        "highest_fold_training_accuracy": training_max,
        "mean_max_validation_accuracy": validation_mean,
        "mean_full_development_training_accuracy": full_dev_train_mean,
        "max_full_development_training_accuracy": full_dev_train_max,
        "final_blend": {
            "cv_weight": FINAL_BLEND_CV_WEIGHT,
            "full_development_weight": FINAL_BLEND_FULLDEV_WEIGHT,
        },
        "oof_metrics": oof_metrics,
        "test_metrics": test_metrics,
        "model_settings": {
            "branch_filters": BRANCH_FILTERS,
            "bigru_units": BIGRU_UNITS,
            "ffn_units": FFN_UNITS,
            "dense_1": DENSE_1,
            "dense_2": DENSE_2,
            "dropout": DROPOUT,
            "spatial_dropout": SPATIAL_DROPOUT,
            "attention_dropout": ATTENTION_DROPOUT,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
        },
    },
    RESULT_DIR
    / "experiment_summary.json",
)

display(summary_df)

print("\nRESULT DIRECTORY")
print(RESULT_DIR)

## V9 run notes

Run in a fresh Colab **v5e-1 TPU** session from top to bottom.

V9 changes two things compared with V8.

First, the fold models are given slightly more fitting room:

```python
WEIGHT_DECAY = 3e-6
SPATIAL_DROPOUT = 0.05
ATTENTION_DROPOUT = 0.05
DROPOUT = 0.12
FOLD_EARLY_STOP_PATIENCE = 14
```

This is intended to move maximum fold training accuracy farther into the
`0.96–0.97` range while preserving a separate best-validation checkpoint.

Second, after five-fold CV, V9 trains three models on **100% of the development
set**:

```python
FINAL_ENSEMBLE_SEEDS = (42, 52, 62)
FINAL_TARGET_TRAIN_ACCURACY = 0.965
FINAL_MIN_EPOCHS = 15
FINAL_MAX_EPOCHS = 24
```

Their predictions are combined with the five best-validation CV models using
a predeclared blend:

```python
FINAL_BLEND_CV_WEIGHT = 0.60
FINAL_BLEND_FULLDEV_WEIGHT = 0.40
```

The independent test is not used to choose these weights, epochs, threshold,
or checkpoints.

The threshold remains:

```python
FIXED_THRESHOLD = 0.50
```

### Important interpretation

The full-development models can legitimately obtain training accuracy similar
to the fold-training accuracy because they are trained on all development data.

Independent-test **accuracy**, however, cannot be forced to equal training
accuracy. If the test result remains materially below the fold training result,
that is a generalization gap, not a training-code failure.

Because earlier versions of this project have already been repeatedly inspected
on the same test set, a publication-final claim should ideally be confirmed on
a newly reserved untouched holdout.